# Tiền xử lý dữ liệu đánh giá Shopee (Shopee Reviews Preprocessing Pipeline)

Notebook này minh họa quy trình tiền xử lý dữ liệu bình luận thu thập được từ Shopee sử dụng lớp xử lý modular **`ShopeeReviewPreprocessor`**.

### Quy trình xử lý bao gồm:
1. **Làm sạch văn bản cơ bản (Basic Cleaning)**: Chuyển chữ thường, loại bỏ ký tự đặc biệt/dấu câu.
2. **Chuẩn hóa Teencode & Viết tắt (Teencode Normalization)**: Tự động tra từ điển `teencode.json` để đồng bộ các cụm từ viết tắt phổ biến trực tuyến (như `ko` -> `không`, `sp` -> `sản phẩm`).
3. **Sửa lỗi chính tả tự động bằng Edit Distance (SymSpell-like/Levenshtein similarity)**: Đối chiếu với từ điển tiếng Việt chuẩn `vietnamese_dictionary.txt` để sửa các lỗi gõ nhầm chữ.
4. **Tách từ tiếng Việt (Word Segmentation)**: Sử dụng gói NLP `underthesea` để tách cụm từ có nghĩa (ví dụ: `sản phẩm` -> `sản_phẩm`).

## 1. Khởi tạo & Kiểm tra thông tin dữ liệu gốc

In [2]:
import pandas as pd
import os

# Đọc dữ liệu thô từ Shopee v2 API Crawler (nằm trong thư mục outputs)
raw_data_path = "../outputs/shopee_reviews_api.csv"
df_raw = pd.read_csv(raw_data_path)

print("---- Thông tin bộ dữ liệu ----")
print(df_raw.info())

print("\n---- Số lượng giá trị Null ----")
print(df_raw.isnull().sum())

print("\n---- Số lượng trùng lặp trùng user và comment ----")
print(df_raw.duplicated(subset=["user", "comment"]).sum())

---- Thông tin bộ dữ liệu ----
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   user              50 non-null     object 
 1   userid            50 non-null     int64  
 2   stars             50 non-null     int64  
 3   product_quality   50 non-null     int64  
 4   seller_service    49 non-null     float64
 5   delivery_service  49 non-null     float64
 6   comment           50 non-null     object 
 7   template_tags     18 non-null     object 
 8   datetime          50 non-null     object 
 9   variant           21 non-null     object 
 10  like_count        50 non-null     int64  
 11  image_count       50 non-null     int64  
 12  video_count       50 non-null     int64  
 13  is_repeated       50 non-null     bool   
 14  cmtid             50 non-null     int64  
dtypes: bool(1), float64(2), int64(7), object(5)
memory usage: 5.6+

## 2. Thực hiện Quy trình Tiền xử lý thông minh (Spelling Correction & Teencode Normalization)

In [3]:
# Import class preprocessor modular đã xây dựng
from preprocess import ShopeeReviewPreprocessor

# Khởi tạo bộ tiền xử lý (tự động nạp teencode.json và tải từ điển tiếng Việt chuẩn)
preprocessor = ShopeeReviewPreprocessor()

# Chạy toàn bộ pipeline tiền xử lý trên dataframe
df_cleaned = preprocessor.process_dataframe(df_raw)

[✓] Custom teencode dictionary loaded: 31 rules.
[✓] Standard vocabulary loaded: 72535 words.
[*] Preprocessing 50 reviews...
[✓] Preprocessing pipeline finished successfully!


## 3. Xem trước kết quả sau tiền xử lý

In [4]:
print("--- So sánh bình luận trước và sau khi tiền xử lý (5 dòng đầu) ---")
for index, row in df_cleaned[['comment', 'comment_corrected', 'tokens_normalized']].head(5).iterrows():
    print(f"[Gốc]: {row['comment']}")
    print(f"[Sửa]: {row['comment_corrected']}")
    print(f"[Token]: {row['tokens_normalized']}")
    print("-" * 50)

--- So sánh bình luận trước và sau khi tiền xử lý (5 dòng đầu) ---
[Gốc]: Giảm mỏi mắt:giảm căng thẳng mắt Độ sáng:2 chế độ màu sáng Thiết kế:thiết kế tinh tế  Sản phẩm tốt, tùy chỉnh được mức độ sáng.
[Sửa]: giảm mỏi mắt giảm căng thẳng mắt độ sáng 2 chế độ màu sáng thiết kế thiết kế tinh tế sản phẩm tốt thùy chỉnh được mức độ sáng
[Token]: giảm mỏi mắt giảm căng_thẳng mắt độ sáng 2 chế_độ màu sáng thiết_kế thiết_kế tinh_tế sản_phẩm tốt thùy_chỉnh được mức_độ sáng
--------------------------------------------------
[Gốc]: Sản phẩm đúng mô tả, đóng gói cẩn thận, giao hàng khá nhanh. Chất lượng ổn trong tầm giá, sử dụng thấy phù hợp nhu cầu. Shop tư vấn nhiệt tình, sẽ ủng hộ thêm lần sau.
[Sửa]: sản phẩm đúng mô tả đóng gói cẩn thận giao hàng khá nhanh chất lượng ổn trong tầm giá sử dụng thấy phù hợp nhu cầu cửa hàng tư vấn nhiệt tình sẽ ủng hộ thêm lần sau
[Token]: sản_phẩm đúng mô_tả đóng_gói cẩn_thận giao hàng khá nhanh chất_lượng ổn trong tầm_giá sử_dụng thấy phù_hợp nhu_cầu cửa_hàng

## 4. Lưu dữ liệu sạch xuống CSV

In [5]:
output_path = "../outputs/shopee_reviews_cleaned.csv"
df_cleaned.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"[✓] Tiền xử lý hoàn tất! Đã lưu dữ liệu sạch vào: {output_path}")

[✓] Tiền xử lý hoàn tất! Đã lưu dữ liệu sạch vào: ../outputs/shopee_reviews_cleaned.csv
